# 3 · Unit commitment (MIP)

**Energy.** A grid operator plans the next 24 hours. For every hour it decides **which power plants are switched on** and **how much each one produces**, so that supply meets demand at the lowest cost.

What makes this hard is the yes/no part:

- a plant that is on must run at least at its **minimum stable output** (it can't run at 3 MW);
- **starting** a plant costs money (fuel to warm up, wear and tear).

Those need **binary variables**: $u_{g,t} = 1$ if plant g is on in hour t. That turns the LP into a **MIP**.

### Set up this notebook

Installs the packages this notebook needs into the running kernel, on Google Colab, Jupyter or VS Code. Already ran `pip install -r requirements.txt`? It just reports they are installed.

In [ ]:
%pip install -q gamspy pandas numpy plotly
# Optional: install your free academic license (otherwise the demo license is used)
# !gamspy install license <your-access-code>

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gamspy as gp

## The data: 5 plants, 24 hours

Costs are in $ per MWh (`mc`) and $ per start (`startup`). Output limits are in MW. `init` says whether the plant is already running at midnight.

In [3]:
units = pd.DataFrame({
    "pmin":    [300, 150,  80,  20,  10],
    "pmax":    [600, 450, 350, 150, 100],
    "mc":      [ 12,  25,  40,  90, 130],
    "startup": [20000, 8000, 3000, 500, 300],
    "init":    [1, 1, 0, 0, 0],
}, index=["nuclear", "coal", "gas_cc", "peaker", "oil"])

hours = [f"h{t:02d}" for t in range(1, 25)]
typical = np.array([720, 690, 670, 660, 670, 720, 820, 930, 1010, 1060, 1090, 1110,
                    1120, 1130, 1150, 1190, 1250, 1340, 1360, 1300, 1180, 1030, 890, 780])
solar = np.array([0, 0, 0, 0, 0, 0, 20, 90, 200, 320, 400, 440,
                  450, 440, 400, 320, 200, 80, 10, 0, 0, 0, 0, 0])
afternoon = (np.arange(24) >= 11) & (np.arange(24) <= 20)
profiles = {
    "typical":  typical,
    "heatwave": np.round(typical * np.where(afternoon, 1.15, 1.03), 1),
    "duck":     typical - solar,          # lots of solar at midday
}
px.line(pd.DataFrame(profiles, index=hours), title="Demand (MW)", width=900, height=350,
        labels={"index": "hour", "value": "MW", "variable": "profile"}).show()
units

,pmin,pmax,mc,startup,init
nuclear,300,600,12,20000,1
coal,150,450,25,8000,1
gas_cc,80,350,40,3000,0
peaker,20,150,90,500,0
oil,10,100,130,300,0


## The model

| | Math | Meaning |
|---|---|---|
| balance | $\sum_g p_{g,t} = D_t$ | supply meets demand every hour |
| min output | $p_{g,t} \ge P^{min}_g\, u_{g,t}$ | if on, at least the minimum |
| max output | $p_{g,t} \le P^{max}_g\, u_{g,t}$ | if off, exactly zero |
| start-up | $v_{g,t} \ge u_{g,t} - u_{g,t-1}$ | a start happens when off → on |
| objective | $\sum_{g,t} mc_g\, p_{g,t} + su_g\, v_{g,t}$ | fuel + start-up cost |

Two GAMSPy features appear here: `t.lag(1)` refers to the previous hour, and `.where[t.first]` adds the midnight status only in the first hour.

In [4]:
def build_uc(demand_profile, problem="MIP"):
    m = gp.Container()
    g = gp.Set(m, description="plants", records=units.index.tolist())
    t = gp.Set(m, description="hours", records=hours)

    pmin = gp.Parameter(m, domain=g, records=units["pmin"].reset_index())
    pmax = gp.Parameter(m, domain=g, records=units["pmax"].reset_index())
    mc   = gp.Parameter(m, domain=g, records=units["mc"].reset_index())
    su   = gp.Parameter(m, domain=g, records=units["startup"].reset_index())
    u0   = gp.Parameter(m, domain=g, records=units["init"].reset_index())
    D    = gp.Parameter(m, domain=t, records=pd.DataFrame({"t": hours, "MW": demand_profile}))

    p = gp.Variable(m, domain=[g, t], type="positive", description="output (MW)")
    u = gp.Variable(m, domain=[g, t], type="binary",   description="1 if plant is on")
    v = gp.Variable(m, domain=[g, t], type="positive", description="1 if plant starts")

    balance = gp.Equation(m, domain=t)
    min_out = gp.Equation(m, domain=[g, t])
    max_out = gp.Equation(m, domain=[g, t])
    startup = gp.Equation(m, domain=[g, t])

    balance[t]    = gp.Sum(g, p[g, t]) == D[t]
    min_out[g, t] = p[g, t] >= pmin[g] * u[g, t]
    max_out[g, t] = p[g, t] <= pmax[g] * u[g, t]
    startup[g, t] = v[g, t] >= u[g, t] - u[g, t.lag(1)] - u0[g].where[t.first]

    cost = gp.Sum((g, t), mc[g] * p[g, t] + su[g] * v[g, t])
    uc = gp.Model(m, equations=m.getEquations(), problem=problem,
                  sense="min", objective=cost)
    return uc, p, u, v

## Solve the typical day

In [5]:
uc, p, u, v = build_uc(profiles["typical"])
uc.solve(options=gp.Options(relative_optimality_gap=0))
print(uc.status, f"total cost: ${uc.objective_value:,.0f}")

ModelStatus.OptimalGlobal total cost: $444,660


In [6]:
def plot_schedule(p, u, title):
    P = p.pivot().reindex(index=units.index, columns=hours).fillna(0)
    U = u.pivot().reindex(index=units.index, columns=hours).fillna(0)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[3, 1.4],
                        vertical_spacing=0.08, subplot_titles=(title, "On/off (darker = on)"))
    colors = ["#30383B", "#788B92", "#F4961A", "#F7C27A", "#B35F00"]
    for unit, color in zip(P.index, colors):
        fig.add_trace(go.Scatter(x=hours, y=P.loc[unit], name=unit, stackgroup="mw",
                                 mode="lines", line_color=color), row=1, col=1)
    fig.add_trace(go.Heatmap(z=U.values, x=hours, y=list(U.index), colorscale="Oranges",
                             zmin=0, zmax=1, showscale=False), row=2, col=1)
    fig.update_yaxes(title_text="MW", row=1, col=1)
    fig.update_yaxes(autorange="reversed", row=2, col=1)
    fig.update_layout(template="simple_white", width=1000, height=600)
    fig.show()

plot_schedule(p, u, "Typical day: who produces what")

## What does integrality cost?

Solve the same model as **RMIP** (the *relaxed* MIP: the binaries may take any value between 0 and 1). It's an LP, so it solves instantly, but look at the on/off values.

In [7]:
uc_r, p_r, u_r, v_r = build_uc(profiles["typical"], problem="RMIP")
uc_r.solve()
print(f"RMIP (relaxed): ${uc_r.objective_value:,.0f}")
print(f"MIP  (real):    ${uc.objective_value:,.0f}")
print(f"Integrality gap: {100 * (uc.objective_value - uc_r.objective_value) / uc.objective_value:.2f}%")
u_r.pivot().reindex(index=units.index, columns=hours).fillna(0).round(2).iloc[:, 8:20]

RMIP (relaxed): $441,477
MIP  (real):    $444,660
Integrality gap: 0.72%


,h09,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20
nuclear,1.0,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
coal,1.0,1.00,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
gas_cc,0.0,0.12,0.5,0.75,0.88,0.89,0.89,0.89,0.89,0.89,0.89,0.71
peaker,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
oil,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


**Discuss:** what does it mean to have the gas plant "0.12 on"? The relaxation is a lower bound on cost. Branch-and-bound (what CPLEX does) closes the gap between that bound and a real schedule. `relative_optimality_gap=0` asks it to prove the best schedule; on big models you would accept, say, 1%.

## Your turn: change the weather

Each table takes one profile: `"heatwave"` or `"duck"` (a sunny day with lots of solar). Solve it, plot it and report back: **total cost, number of start-ups, and which plants switch on.**

In [8]:
profile = "heatwave"          # try "heatwave" or "duck"

uc2, p2, u2, v2 = build_uc(profiles[profile])
uc2.solve(options=gp.Options(relative_optimality_gap=0))
print(f"{profile}: total cost ${uc2.objective_value:,.0f}, start-ups: {v2.records['level'].sum():.0f}")
plot_schedule(p2, u2, f"{profile.capitalize()} day")

heatwave: total cost $548,619, start-ups: 3


**Stretch**

1. **Minimum up-time.** Coal can't be switched off within 6 hours of starting. Add the constraint $\sum_{\tau = t-5}^{t} v_{coal,\tau} \le u_{coal,t}$. (Hint: use an `Alias` of `t` and a condition on `ord`.)
2. **Reserve.** The operator wants at least 10% of demand as spare capacity from plants that are on: $\sum_g P^{max}_g u_{g,t} \ge 1.1\, D_t$. How much does that security cost?

In [9]:
# Your work here
